<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.1-stationary-heat/Ex08.1_pinn_plate_with_channel.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.1 — A Heated Plate with a Cooling Channel

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L8.1 · Stationary Heat**

An aluminium plate generates heat and is cooled by a liquid that runs through a
channel at its centre. How hot does the plate get, and where? It is steady
conduction, the Poisson equation of L7 with real units - and a domain that is
no longer a rectangle. You solve it three ways: a finite-element reference on
a mesh fitted to the channel, finite differences on a square grid, and a
physics-informed network with the channel built in. Then you compare them on
accuracy and on time.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and finite differences on a mesh for an agreed 0.10 K | — |
| **5** | a PINN with the channel wall built in, and the comparison | the trial function, the residual |
| **6** | what the comparison says | — |
| **7 – 8** | your report, and the mini project proposal | your answers |

About four minutes on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.1-stationary-heat/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex08.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

A 100 × 100 mm plate of aluminium carries power electronics and generates heat
evenly through its volume. A coolant at 40 °C runs through an elliptical
channel at its centre and holds the channel wall at that temperature (the
channel is the hole of the lecture's slides). The
outer edges are insulated, so every watt generated must leave through the
channel. What is the temperature at every point of the plate, and how hot is
its hottest point?

## 2 · The data

| | |
|---|---|
| plate | 100 × 100 mm, aluminium alloy 6061, $k$ = 167 W/(m·K) |
| heat generated | $Q$ = 1 MW/m³, evenly |
| cooling channel | an ellipse of 36 × 22 mm at the centre, its wall held at 40 °C |
| outer edges | insulated: no heat crosses them |

## 3 · The physics

Of the three mechanisms of L8.1 only **conduction** is solved. The coolant's
convection is not: it enters as the fixed temperature of the channel wall.
Fourier's law in an energy balance gives the Poisson equation

$$k\,(T_{xx} + T_{yy}) + Q = 0, \qquad T = 40\ ^\circ\text{C on the channel}, \qquad \frac{\partial T}{\partial n} = 0 \text{ on the edges}$$

It is **elliptic**: there is no time, and every point feels every wall at
once. One wall holds a temperature, so the solution is unique.

**Scaled units.** Lengths are divided by the side $L$ and the rise above the
coolant by $\Delta T = QL^2/k$ = 59.88 K. The scaled rise $\theta$ obeys

$$\theta_{xx} + \theta_{yy} + 1 = 0, \qquad \theta = 0 \text{ on the channel}, \qquad \frac{\partial\theta}{\partial n} = 0 \text{ on the edges}$$

on the unit square, with every term of order one, and $T = 40 + 59.88\,\theta$.
Everything below is solved in these units and printed in kelvin.

**The geometry is a formula.** The level set

$$\phi(x, y) = \left(\frac{x - 0.5}{0.18}\right)^2 + \left(\frac{y - 0.5}{0.11}\right)^2 - 1$$

is negative inside the channel, zero on its wall and positive in the plate.
The cell below prints the data and draws the plate.

In [ ]:
pb.describe_problem()
xy = pb.sample_plate_with_hole(500)                         # points in the plate: kept where phi > 0
wall = pb.sample_ellipse_boundary(60)                       # points on the channel wall, equal arc length
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.plot(xy[:, 0] * 100, xy[:, 1] * 100, ".", ms=3, color="0.55", label="points in the plate")
ax.plot(wall[:, 0] * 100, wall[:, 1] * 100, "o", ms=3, color="tab:orange", label="channel wall, 40 °C")
ax.plot([0, 100, 100, 0, 0], [0, 0, 100, 100, 0], "--", color="tab:blue", label="insulated edges")
ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); ax.set_ylabel("y  [mm]")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
plt.tight_layout(); plt.show()

**What you should see.** The data of section 2, a temperature scale of 59.88 K
and 9378 W per metre of depth to remove; and the plate with its channel, the
wall points evenly spaced along the ellipse.

---

## 4 · The reference, and finite differences

**Two classical methods, and what differs between them.** Both replace the
field by its values at a finite number of nodes and end in one linear system.
They differ in what they ask of the equation.

* **Finite differences** keep the equation as it stands and replace every
  derivative by a difference between neighbouring nodes of a **square grid**
  of spacing $h$:
  $\theta_{xx} \approx (\theta_{i+1,j} - 2\theta_{i,j} + \theta_{i-1,j})/h^2$.
  The equation is asked to hold **at every node**. The grid cannot follow a
  curved wall: a node is either in the channel or in the plate, so the wall
  becomes a **staircase**.
* **Finite elements** cut the plate into **triangles**. The unknowns are the
  values at the corners, and inside a triangle the field is a flat facet. The
  equation is not asked to hold at points but **on average**: it is multiplied
  by a small tent-shaped weight round each node and integrated over the plate,
  $\int \nabla\theta\cdot\nabla v\,dA = \int v\,dA$ for the tent $v$ of every
  node. Triangles can be laid along any curve, so **nodes sit on the wall**,
  and an insulated edge needs nothing: it is what the integral gives when
  nothing is prescribed.

The physics-informed network of section 5 is a third way: one smooth formula
for the whole plate, the equation asked at scattered points, and an optimiser
in place of the linear solve.

The cell below draws both meshes, coarse enough to see.

In [ ]:
coarse = pb.reference(1 / 25)                                # the fitted mesh, coarse enough to see
g = np.linspace(0, 1, 26); MX, MY = np.meshgrid(g, g)        # the square grid at the same spacing
held = pb.ellipse_phi(np.c_[MX.ravel(), MY.ravel()]).reshape(MX.shape) <= 0   # nodes the grid puts in the channel
a = np.linspace(0, 2 * np.pi, 200)
wall_x, wall_y = 100 * (pb.HOLE["xc"] + pb.HOLE["a"] * np.cos(a)), 100 * (pb.HOLE["yc"] + pb.HOLE["b"] * np.sin(a))
fig, ax = plt.subplots(1, 2, figsize=(10.5, 5.0))
ax[0].triplot(coarse["P"][:, 0] * 100, coarse["P"][:, 1] * 100, coarse["tri"], lw=0.5, color="0.45")
ax[0].set_title("finite elements: triangles fitted to the wall", fontsize=10)
ax[1].plot(MX * 100, MY * 100, lw=0.5, color="0.45"); ax[1].plot(MX.T * 100, MY.T * 100, lw=0.5, color="0.45")
ax[1].plot(MX[held] * 100, MY[held] * 100, "s", ms=4, color="tab:blue", label="nodes held at 40 °C: the staircase")
ax[1].set_title("finite differences: a square grid", fontsize=10); ax[1].legend(frameon=False, fontsize=8, loc="upper right")
for q in ax:
    q.plot(wall_x, wall_y, color="tab:orange", lw=2)           # the true wall of the channel
    q.set_aspect("equal"); q.set_xlim(20, 80); q.set_ylim(25, 75); q.set_xlabel("x  [mm]")
ax[0].set_ylabel("y  [mm]"); plt.tight_layout(); plt.show()

**What you should see.** The same part of the plate, twice, at a spacing of
4 mm. On the left the triangles follow the orange wall: their nodes lie on it.
On the right the wall cuts through the squares, and the blue nodes are the
ones the grid holds at 40 °C: the channel it solves for has corners, and is
misplaced by up to one spacing.

**The reference.** This problem has no exact solution, so the methods are
scored against a **converged numerical solution**: the finite elements of the
left picture, `pb.reference(spacing)`. Its accuracy is shown the usual way:
the node spacing is halved until the answer stops moving. The cell below runs
four meshes and prints how far each is from the next.

In [ ]:
g = np.linspace(0, 1, 201); GX, GY = np.meshgrid(g, g)       # the points every answer is compared on
plate = pb.ellipse_phi(np.c_[GX.ravel(), GY.ravel()]).reshape(GX.shape) > 0   # True in the plate
fields = {}
for n in (50, 100, 200, 400):
    t0 = time.perf_counter(); r = pb.reference(1 / n); sec = time.perf_counter() - t0
    fields[n] = (r, pb.kelvin(pb.reference_at(r, GX[plate], GY[plate])))
    print(f"  spacing L/{n}: {len(r['P']):6d} nodes, hottest point {pb.T_COOLANT + pb.kelvin(r['theta'].max()):.2f} °C, {sec:.2f} s")
for a, b in ((50, 100), (100, 200), (200, 400)):
    print(f"  L/{a} against L/{b}: worst difference {np.nanmax(np.abs(fields[a][1] - fields[b][1])):.2e} K")
ref, T_ref = fields[200]                                     # the reference from here on
e_ref = np.nanmax(np.abs(fields[200][1] - fields[400][1]))   # how far the reference is from the next finer mesh
t_ref = middle(lambda: pb.reference(1 / 200), reps=3)

**What you should see.** The hottest point settling at 49.56 °C - a rise of
9.56 K, in the corners, the points furthest from the channel - and the
difference between successive meshes falling with every halving: 6.04e-02,
1.82e-02 and 3.29e-03 K. The mesh with spacing L/200 (38 009 nodes) is within
0.004 K of the next one, **thirty times better than the accuracy agreed
below**, so it is the reference. It takes under a second.

**Finite differences.** The square grid of the right picture: the five-point
stencil at every node of the plate, the insulated edges by a mirrored node,
and every node inside the channel held at 40 °C. **The agreed accuracy is 0.10 K**; the
cell below finds the coarsest grid that meets it.

In [ ]:
def fdm_plate(n):
    # n x n nodes on the unit square; returns the node coordinates and the scaled rise
    h = 1.0 / (n - 1)
    g = np.linspace(0, 1, n); X, Y = np.meshgrid(g, g)
    D = sp.diags([1.0, -2.0, 1.0], [-1, 0, 1], shape=(n, n)).tolil()
    D[0, 1] = 2.0; D[-1, -2] = 2.0                             # a mirrored node: zero gradient across the edge
    D = D.tocsr() / h ** 2
    A = (sp.kron(sp.identity(n), D) + sp.kron(D, sp.identity(n))).tocsr()   # the five-point Laplacian
    free = np.where(pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).ravel() > 0)[0]   # nodes in the plate
    theta = np.zeros(n * n)                                    # nodes in the channel stay at zero: the staircase
    theta[free] = spla.spsolve(A[free][:, free].tocsc(), -np.ones(len(free)))
    return X, Y, theta.reshape(n, n)

TARGET = 0.10                                                 # K: the agreed accuracy
for n in (81, 161, 321, 641):
    X, Y, th = fdm_plate(n)
    m = pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).reshape(X.shape) > 0
    err = np.nanmax(np.abs(pb.kelvin(th[m]) - pb.kelvin(pb.reference_at(ref, X[m], Y[m]))))
    sec = middle(lambda: fdm_plate(n), reps=3)
    print(f"  {n:3d} x {n} nodes: worst error {err:.2e} K, one solve {sec*1e3:.0f} ms"
          + ("   <- meets 0.10 K" if err <= TARGET else ""))
N_FDM = 641                                                   # the coarsest grid that meets it

**What you should see.** The error falling slowly - 4.69e-01, 3.19e-01,
1.27e-01 and 6.43e-02 K - at best halving when the spacing halves. The
staircase is the reason: the wall is misplaced by up to one spacing. It takes
641 × 641 nodes, about 400 000 unknowns and a few seconds, to reach 0.10 K.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(x, y)$ and returns the scaled rise. It is four layers
of 32 tanh neurons, trained on 2000 collocation points in the plate.

**The channel wall is built in** (L8.1's **Boundary Multiplier** slide). The
trial function

$$\hat\theta(x, y) = S\,\phi(x, y)\,\mathcal{N}(x, y)$$

is exactly zero on the channel wall whatever the network does, because $\phi$
is. This is **hard enforcement**: the condition holds by construction, for any
weights, and needs no loss term.

$S$ = 0.01 is a fixed number, not trained, and it is there for size only.
$\phi$ grows to 27 in the corners while the answer $\theta$ stays below 0.2;
without $S$ the network would have to return numbers of a few thousandths,
and with it numbers of order one, which is what a network trains well at.

**The insulated edges stay a loss term**, with a weight - **soft
enforcement** - because a multiplier can hold a value, not a derivative. On each edge the gradient along the normal is
asked to be zero, and the weight $w_N$ = 10 sets how much that counts against
the residual.

**TODO 1** is the trial function; **TODO 2** the residual of the scaled
equation, $\hat\theta_{xx} + \hat\theta_{yy} + 1$.

In [ ]:
SCALE = 0.01                                                  # the size of the answer over the size of phi
class Plate(torch.nn.Module):                                 # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_hidden=32, n_layers=4)       # dense tanh network: x, y in
    def forward(self, xy):
        # TODO 1: the trial function - SCALE times the multiplier phi times the network.
        #
        #   return SCALE * pb.hole_multiplier(xy) * self.net(xy)
        raise NotImplementedError("Write the trial function")

In [ ]:
# TODO 2: the residual of the scaled equation, theta_xx + theta_yy + 1.
#
#   def residual(model, xy):
#       th = model(xy)
#       return d2(th, xy, 0) + d2(th, xy, 1) + 1.0
raise NotImplementedError("Write residual(model, xy)")

In [ ]:
W_N = 10.0                                                    # the weight of the insulated edges
xy_f = to_tensor(pb.sample_plate_with_hole(2000), requires_grad=True)   # collocation points, differentiable
edges = [(to_tensor(p, requires_grad=True), axis) for p, axis in pb.sample_outer_edges(60)]   # 60 points on each edge

def edge_loss(model):
    # the gradient along the normal, squared, averaged over the four edges
    return sum(grad(model(p), p)[:, axis:axis + 1].pow(2).mean() for p, axis in edges) / 4

model = Plate().to(DEVICE)                                    # four layers of 32
describe(model.net, n_collocation=len(xy_f))                  # the parameter count and the collocation-to-parameter ratio
loss_fn = lambda: residual(model, xy_f).pow(2).mean() + W_N * edge_loss(model)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
history = train_two_stage(model, loss_fn, adam_steps=500, lbfgs_steps=150, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s; residual {residual(model, xy_f).pow(2).mean().item():.2e}, "
      f"edges {edge_loss(model).item():.2e}")

In [ ]:
X, Y, th_fdm = fdm_plate(N_FDM)                               # the finite-difference answer
m = pb.ellipse_phi(np.c_[X.ravel(), Y.ravel()]).reshape(X.shape) > 0
T_r = pb.kelvin(pb.reference_at(ref, X[m], Y[m]))             # the reference on the same nodes
Pn = to_tensor(np.c_[X[m], Y[m]])                             # the same nodes for the network
def pinn_field():
    with torch.no_grad():  # no gradient tracking: evaluation only
        return pb.kelvin(to_numpy(model(Pn)).ravel())
T_f, T_p = pb.kelvin(th_fdm[m]), pinn_field()
e_f, e_p = np.abs(T_f - T_r), np.abs(T_p - T_r)
t_fdm = middle(lambda: fdm_plate(N_FDM), reps=3)
t_pin = middle(pinn_field)
balance = pb.heat_out(model) / pb.plate_area()                # Fourier's flux round the wall, over the heat generated

# "time to the answer" counts everything from the problem to the field: for the network, its training
print(f"{'':32s}{'unknowns':>10s}{'hottest':>10s}{'worst error':>13s}{'time to the answer':>20s}")
print(f"  {'reference, fitted L/200':30s}{len(ref['P']):10d}{pb.T_COOLANT + np.nanmax(T_r):8.2f} °C{e_ref:11.2e} K{t_ref:18.2f} s   (error: against the next finer mesh)")
print(f"  {'finite differences, 641 x 641':30s}{int(m.sum()):10d}{pb.T_COOLANT + T_f.max():8.2f} °C{np.nanmax(e_f):11.2e} K{t_fdm:18.2f} s")
print(f"  {'PINN, 4 x 32':30s}{parameter_count(model):10d}{pb.T_COOLANT + T_p.max():8.2f} °C{np.nanmax(e_p):11.2e} K{train_time + t_pin:18.2f} s   (training; the trained network then gives the field in {t_pin * 1e3:.0f} ms)")
print(f"energy balance of the PINN: heat out / heat generated = {100 * balance:.2f} %")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
F = np.full(X.shape, np.nan)
for ax, vals, title, cmap in ((axes[0], pb.T_COOLANT + T_r, "reference  [°C]", "inferno"),
                              (axes[1], e_f, "finite differences: error  [K]", "viridis"),
                              (axes[2], e_p, "PINN: error  [K]", "viridis")):
    F[m] = vals
    c = ax.pcolormesh(X * 100, Y * 100, F, cmap=cmap, shading="auto")
    ax.set_title(title); ax.set_aspect("equal"); ax.set_xlabel("x  [mm]"); plt.colorbar(c, ax=ax, shrink=0.85)
axes[0].set_ylabel("y  [mm]")
plt.tight_layout(); plt.savefig(cc.output_path("Ex08.1_report.png"), dpi=150); plt.show()

**What you should see.** The reference and the network put the hottest point
at 49.56 °C, in the corners; finite differences at 49.60 °C. The network's
energy balance is 99.51 %: the heat it sends through the channel wall is the
heat generated, to half a per cent.

**By accuracy** the order is reference (within about 0.003 K of the next
finer mesh), network (about 0.018 K), finite differences (0.064 K). **By time
to the answer** it is reference (about half a second), finite differences
(about two to three seconds), network (one to two minutes, all of it training). A
trained network gives the field in a few tens of milliseconds, but that is
reading an answer already paid for, not computing one.

**Why the network is more exact than finite differences, with a hundred times
fewer unknowns.** Its wall is in the right place. The field is smooth, and a
smooth field is cheap to describe: about 3 300 weights are enough. Finite
differences spend 400 000 unknowns and still solve for a channel with
corners. The reference has its nodes on the wall too, and beats both.

---

## 6 · What the comparison says

* **All three meet the agreed 0.10 K**, and they agree on the number an
  engineer wants: the hottest point is 9.56 K above the coolant, give or take
  a few hundredths.
* **The curved wall is what finite differences pay for.** On a rectangle they
  gain a factor of four with every halving; on the staircase they gain two at
  best, and need 400 000 unknowns. The fitted reference needed 38 009.
* **The network does not see the staircase.** The channel is a formula, built
  into the trial function, so the wall is exact for any size of network. That
  is the reason to use the level set.
* **For one plate, solved once, the classical methods win on time.** In the
  table the reference is the most exact and has the shortest time to the
  answer; the network is second in accuracy and last in time, by a factor of
  more than a hundred. Its training buys nothing here. It starts to pay when a
  wall cannot be meshed easily, when the same network must answer for many
  designs, or when a number of the model is unknown and has to be found from
  measurements, as in Ex_08.2.
* **The energy balance is the check that needs no reference.** On a design
  with no reference solution it is what tells you the flux wall was met.

---

## 7 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex08.1_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L8.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The channel wall is held at a temperature and the outer edges are insulated. Which kind of boundary condition is each, and what would change in the model if the coolant were described by convection, with a coefficient h, instead? (-> L8.1 Q1)": """
""",
    "Why is the plate solved for the scaled rise and not for the temperature in °C, and what does the temperature scale Q L²/k = 59.88 K tell you before any solver has run? (-> L8.1 Q2)": """
""",
    "What does the multiplier guarantee on the channel wall, and why did finite differences need 641 × 641 nodes for 0.10 K when the fitted reference needed 38 009? (-> L8.1 Q3)": """
""",
    "The insulated edges are a loss term with the weight 10. What did the energy balance and the two loss terms tell you about how well the edges and the equation were met? (-> L8.1 Q4)": """
""",
    "To conclude: for this one plate, solved once, which of the three methods would you choose, and what would have to change for the network to be worth its training? (-> L8.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | unknowns | hottest point | worst error | time to the answer |", "|---|---|---|---|---|",
            f"| reference, fitted mesh L/200 | {len(ref['P'])} | {pb.T_COOLANT + np.nanmax(T_r):.2f} °C | {e_ref:.2e} K (against the next finer mesh) | {t_ref:.2f} s |",
            f"| finite differences, 641 x 641 | {int(m.sum())} | {pb.T_COOLANT + T_f.max():.2f} °C | {np.nanmax(e_f):.2e} K | {t_fdm:.2f} s |",
            f"| PINN, 4 x 32 | {parameter_count(model)} | {pb.T_COOLANT + T_p.max():.2f} °C | {np.nanmax(e_p):.2e} K | {train_time + t_pin:.2f} s, all but {t_pin*1e3:.0f} ms of it training |",
            "", f"Energy balance of the PINN: heat out / heat generated = {100 * balance:.2f} %"]
    out = ["# Ex_08.1 — A Heated Plate with a Cooling Channel", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex08.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex08.1_report.md into Ex08.1_report.pdf, with any figure
# saved as Ex08.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex08.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex08.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex08.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex08.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 8 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP8.1A · Cooled by a fluid, through several bores

**The problem.** The hole of this set was held at a fixed temperature. A real cooling bore is
not: the fluid takes heat away in proportion to how much hotter the wall is
than the fluid - a Robin condition, $-k\,\partial T/\partial n = h\,(T - T_c)$.
Here a block has three bores, each with its own fluid, in real units.

| | |
|---|---|
| section | 100 × 100 mm of aluminium, 167 W/m·K, generating 1 MW/m³, outer edges insulated |
| bores 1, 2 | water, radius 8 mm, centred at (30, 30) and (70, 30) mm, $h$ = 4000 W/m²K, 40 °C |
| bore 3 | oil, an ellipse 12 × 6 mm (semi-axes), centred at (50, 72) mm, $h$ = 500 W/m²K, 40 °C |

The conductivity and the two heat transfer coefficients are typical values,
sourced before the data is released.

**Why a plain PINN is not enough.** A Robin condition cannot be built into the network the way the fixed
temperature of this set was: it ties the value to the slope, so it has to be a
loss term, on curved walls where the normal changes at every point, with a
weight chosen against the residual as in section 5 - and three bores that
carry very different shares of the heat.

**What you build.** A PINN for $T(x, y)$ with a weighted Robin term on every bore wall, sampled by
arc length, in scaled units. You show from the per-term losses that every wall
was learned, and you check the result with the heat balance.

**The ground truth you get.** `MP8.1A_truth.npz`, from `tools/miniprojects/ex081_truth.py`: linear finite
elements on a Delaunay mesh with the bores cut out, nodes 0.5 mm apart
(38,033 nodes), the temperature at every node and the heat through each bore.
It passes two checks: with $h$ = 10⁹ the walls sit within 1.1e-04 K of their
fluid, and the heat through the bores equals the heat made.

**Worked example.** 

| node spacing | nodes | hottest point | water bores (each) | oil bore | made |
|---|---|---|---|---|---|
| 2 mm | 2,470 | 74.69 °C | 4241.3 W/m | 894.6 W/m | 9377.2 W/m |
| 1 mm | 9,659 | 74.66 °C | 4239.1 W/m | 894.9 W/m | 9373.1 W/m |
| 0.5 mm | 38,033 | 74.66 °C | 4238.6 W/m | 894.9 W/m | 9372.0 W/m |

The two water bores take 90 % of the heat; the oil bore, with an eighth of
their $h$, takes 10 %.

**Requirements.**

* The hottest point within **0.2 K**.
* The temperature within **0.5 K** everywhere.
* The heat through each bore within **2 %**, and your own heat balance within **1 %**.

**What you hand in.** The error map against the ground truth, the heat through each bore against the
ground truth's, your heat balance, and the per-term loss histories. Say which
bore the network found hardest, and why.

### MP8.1B · The cold plate in three dimensions

**The problem.** The plate of this set was a section: nothing changed along the channel, so
two coordinates were enough. A real cold plate is heated from above by a chip
that covers a small part of it, and the heat spreads sideways and downwards at
once before it reaches the coolant. That takes three coordinates. The
question is the designer's: how hot is the chip, and how much of that is the
spreading?

| | |
|---|---|
| block | 100 × 100 mm in plan, 10 mm thick, aluminium, 167 W/m·K |
| chip | 150 W on 20 × 20 mm of the top face, from (25, 50) to (45, 70) mm: 375 kW/m² |
| cooling | the bottom face, a liquid at 40 °C with $h$ = 5000 W/m²K |
| other faces | insulated: the four sides, and the top beside the chip |

The conductivity is the set's. The heat transfer coefficient and the chip's
power are typical values, sourced before the data is released.

**Why a plain PINN is not enough.** Three inputs in place of two, and second derivatives in three directions. The flux
is not smooth: it is 375 kW/m² under the chip and zero beside it, so the
steepest part of the field sits along the chip's edge, on a boundary. And two
of the walls are conditions on the slope, which cannot be built into the
trial function as the set's fixed temperature was.

**What you build.** A PINN for $T(x, y, z)$ in scaled units, with the chip's flux and the Robin
condition of the bottom as weighted loss terms and the insulated faces as a
third. You choose where to put the points - the chip's edge needs more - and
check the result with the heat balance: 150 W must leave through the bottom.

**The ground truth you get.** `MP8.1B_truth.npz`, from `tools/miniprojects/ex081_truth.py`: the exact
solution as a double cosine series (`coldplate3d.py`), 400 × 400 terms, on
201 × 201 points at eleven heights. It is checked against a second,
independent series, which rebuilds it to 1.5e-05 K.

**Worked example.** 

| terms | hottest point | where, on the top | bottom face | heat out |
|---|---|---|---|---|
| 50 × 50 | 66.47 °C | (35.0, 60.0) mm | 40.34 to 53.91 °C | 150.00 W |
| 100 × 100 | 66.34 °C | (34.5, 60.5) mm | 40.34 to 53.91 °C | 150.00 W |
| 400 × 400 | 66.35 °C | (35.0, 60.0) mm | 40.34 to 53.91 °C | 150.00 W |

The chip sits **26.35 K** above the coolant. If the heat spread evenly the
bottom would be 3.00 K up, and with no spreading at all the chip would be
75 K up: the spreading is most of the answer.

**Requirements.**

* The hottest point within **0.5 K**.
* The temperature within **1 K** everywhere in the block.
* The heat through the bottom face within **2 %** of 150 W, from your own network.

**What you hand in.** The error against the ground truth on the top face, the bottom face and one
vertical cut through the chip, your heat balance, the per-term loss histories,
and how the error changed when you moved points to the chip's edge.